## Introduction to Apache Spark and PySpark

In [ ]:
# Import SparkSession from pyspark.sql
from pyspark.sql import SparkSession

# Create my_spark
my_spark = SparkSession.builder.appName("my_spark").getOrCreate()

# Print my_spark
print(my_spark)

In [ ]:
# Read in the CSV
census_adult = my_spark.read.csv('data/adults.csv')
# Show the DataFrame
census_adult.show()

### PySpark DataFrames

In [ ]:
# Load the CSV file into a DataFrame
salaries_df = my_spark.read.csv("data/salaries.csv", header=True, inferSchema=True)

# Count the total number of rows
row_count = salaries_df.count()
print(f"Total rows: {row_count}")

# Group by company size and calculate the average of salaries
salaries_df.groupBy("company_size").agg({"salary_in_usd": "avg"}).show()
salaries_df.show()

In [ ]:
ca_salaries_df = salaries_df.filter(salaries_df['company_location'] == "CA")

In [ ]:
# Average salary for entry level in Canada
CA_jobs = ca_salaries_df.filter(ca_salaries_df['company_location'] == "CA").filter(ca_salaries_df['experience_level']
== "EN").groupBy().avg("salary_in_usd")

# Show the result
CA_jobs.show()

In [ ]:
# Load the dataframe
census_df = my_spark.read.json("data/adults.json")

# Filter rows based on age condition
salary_filtered_census = census_df.filter(census_df['age'] > 40)

# Show the result
salary_filtered_census.show()

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Fill in the schema with the columns you need from the exercise instructions
schema = StructType([StructField("age", IntegerType()),
                    StructField("education_num", IntegerType()),
                    StructField("marital_status",StringType()),
                    StructField("occupation", StringType()),
                    StructField("income", StringType()),
                    ])

# Read in the CSV, using the schema you defined above
census_adult = my_spark.read.csv("data/adults.csv", sep=',', header=False, schema=schema)

# Print out the schema
census_adult.printSchema()

## PySpark in Python

### Data Manipulation with DataFrames

In [ ]:
census_df = salaries_df

# Drop rows with any nulls
census_cleaned = census_df.na.drop()

# Show the result
census_cleaned.show()

In [ ]:
# Create a new column 'weekly_salary'
census_df_weekly = census_df.withColumn('weekly_salary', census_df['salary_in_usd'] / 52)

# Rename the 'age' column to 'years'
census_df_weekly = census_df_weekly.withColumnRenamed('work_year', 'year')

# Show the result
census_df_weekly.show()

### Advanced DataFrame Operations

In [ ]:
airports = my_spark.read.csv("data/airports.csv",header=True, inferSchema=True).drop("_c0")
flights = my_spark.read.csv("data/flights.csv", header=True, inferSchema=True).drop("_c0")

airports.show()
flights.show()

In [ ]:
# Examine the data
airports.show()

# .withColumnRenamed() renames the "faa" column to "dest"
airports = airports.withColumnRenamed("faa", "dest")

# Join the DataFrames
flights_with_airports = flights.join(airports, "dest", "left_outer")

# Examine the new DataFrame
flights_with_airports.show()

### User Defined Functions (UDFs)

In [ ]:
census_df = census_adult
census_df.show()

In [ ]:
from pyspark.sql.functions import udf
age_category = lambda age: "young" if age < 30 else "middle-aged" if age < 60 else "senior"
age_category_df = census_df.select("age").distinct()

# Register the function age_category as a UDF
age_category_udf = udf(age_category, StringType())

# Apply your udf to the DataFrame
age_category_df_2 = age_category_df.withColumn("category", age_category_udf(age_category_df["age"]))

# Show df
age_category_df_2.show()

## Introduction to PySpark SQL

In [ ]:
# Import SparkSession from pyspark.sql
from pyspark.sql import SparkSession

# Create my_spark
spark = SparkSession.builder.appName("my_spark").getOrCreate()

# Print my_spark
print(spark)

### Resilient Distributed Datasets (RDDs)    

In [ ]:
# Create a DataFrame
df_salaries = spark.read.csv("data/salaries.csv", header=True, inferSchema=True)

# Convert DataFrame to RDD
rdd = df_salaries.rdd

# Show the RDD's contents
rdd.collect()
print(rdd)

In [ ]:
# Create an RDD from the df_salaries
rdd_salaries = df_salaries.rdd

# Collect and print the results
print(rdd_salaries.collect())

# Group by the experience level and calculate the maximum salary
dataframe_results = df_salaries.groupby("experience_level").agg({"salary_in_usd": 'max'})

# Show the results
dataframe_results.show()

### Spark SQL

In [ ]:
# Generate data
# EmployeeID: string, Name: string, Department: string, Position: string, Salary: int, CompanyCode: string, CountryCode: string]
columns = ["EmployeeID", "Name", "Department", "Position", "Salary", "CompanyCode", "CountryCode"]
data = [
    ("001", "John Doe", "Engineering", "Software Engineer", 75000, "COMP001", "US"),
    ("002", "Jane Smith", "Marketing", "Marketing Manager", 85000, "COMP001", "TR"),
    ("003", "Bob Johnson", "Sales", "Marketing Manager", 65000, "COMP001", "US"),
    ("004", "Alice Williams", "HR", "HR Specialist", 70000, "COMP001", "US"),
    ("005", "Charlie Brown", "Finance", "Financial Analyst", 80000, "COMP001", "CA"),
    ("006", "Emily Davis", "Engineering", "HR Specialist", 90000, "COMP001", "US"),
    ("007", "David Wilson", "Marketing", "Content Strategist", 78000, "COMP001", "UK"),
    ("008", "Sophia Martinez", "Sales", "Account Executive", 72000, "COMP001", "US"),
    ("009", "James Anderson", "HR", "Recruiter", 68000, "COMP001", "US"),
    ("010", "Olivia Thomas", "Finance", "Software Engineer", 95000, "COMP001", "SW")
]

df = spark.createDataFrame(data, columns)
df.show()

In [ ]:
# Register as a view
df.createOrReplaceTempView("data_view")

# Advanced SQL query: Calculate total salary by Position
result = spark.sql("""
    SELECT Position, SUM(Salary) AS Total_Salary
    FROM data_view
    GROUP BY Position
    ORDER BY Total_Salary DESC
    """
)
result.show()

In [ ]:
# Create a temporary table "people"
df.createOrReplaceTempView("people")

# Select the names from the temporary table people
query = """SELECT name FROM people"""

# Assign the result of Spark's query to people_df_names
people_df_names = spark.sql(query)

# Print the top 10 names of the people
people_df_names.show(10)

In [ ]:
salaries_df = spark.read.csv("data/salaries.csv", header=True, inferSchema=True)
salaries_df.show()

In [ ]:
# Create a temporary view of salaries_table
salaries_df.createOrReplaceTempView('salaries_table')

# Construct the "query"
query = '''SELECT job_title, salary_in_usd FROM salaries_table WHERE company_location == "CA"'''

# Apply the SQL "query"
canada_titles = spark.sql(query)

# Generate basic statistics
canada_titles.describe().show()

### PySpark Aggregations

In [ ]:
# Find the minimum salaries for small companies
salaries_df.filter(salaries_df.company_size == "S").groupBy().min("salary_in_usd").show()

# Find the maximum salaries for large companies
salaries_df.filter(salaries_df.company_size == "L").groupBy().max("salary_in_usd").show()

In [ ]:
# DataFrame Creation
data = [("HR", "3000"), ("IT", "4000"), ("Finance", "3500")]
columns = ["Department", "Salary"]
df = spark.createDataFrame(data, schema=columns)

# Map the DataFrame to an RDD
rdd = df.rdd.map(lambda row: (row["Department"], row["Salary"]))

# Apply a lambda function to get the sum of the DataFrame
rdd_aggregated = rdd.reduceByKey(lambda x, y: x + y)

# Show the collected Results
print(rdd_aggregated.collect())

In [ ]:
# Average salaries at large us companies
large_companies=salaries_df.filter(salaries_df.company_size == "L").filter(salaries_df.company_location == "US").groupBy().avg("salary_in_usd")

#set a large companies variable for other analytics
large_companies=salaries_df.filter(salaries_df.company_size == "L").filter(salaries_df.company_location == "US")

# Total salaries in usd
large_companies.groupBy().sum("salary_in_usd").show()

### PySpark at Scale

In [ ]:
data = [('Alice', 'HR', 3000), ('Bob', 'IT', 4000), ('Cathy', 'HR', 3500)]
columns = ["Name", "Department", "Salary"]

In [ ]:
# Import SparkSession from pyspark.sql
from pyspark.sql import SparkSession

# Create my_spark
my_spark = SparkSession.builder.appName("final_spark").getOrCreate()

# Print my_spark
print(my_spark)

# Load dataset into a DataFrame
df = my_spark.createDataFrame(data, schema=columns)

df.show()

In [ ]:
# Cache the DataFrame
df.cache()

# Change salary column to integer type
from pyspark.sql.types import IntegerType
df = df.withColumn("Salary", df["Salary"].cast(IntegerType()))

# Perform aggregation
agg_result = df.groupBy("Department").sum("Salary")
agg_result.show()

# Analyze the execution plan
agg_result.explain()

# Uncache the DataFrame
df.unpersist()